# 🎥 Notebook 6: WebRTC Overview

WebRTC enables **peer-to-peer** communication directly between browsers! Perfect for video calls, screen sharing, and reducing server load.

## Learning Objectives

By the end of this notebook, you'll understand:
- How WebRTC enables peer-to-peer connections
- The signaling process and STUN/TURN servers
- When to use WebRTC in system design
- Trade-offs compared to server-mediated approaches

**Note:** This is a conceptual overview. WebRTC is complex and usually requires specialized libraries in production.

## 🤔 What is WebRTC?

WebRTC (Web Real-Time Communication) allows browsers to communicate **directly** with each other!

```
Traditional (Server-Mediated):       WebRTC (Peer-to-Peer):
                                     
┌────────┐      ┌────────┐           ┌────────┐     ┌────────┐
│ Alice  │─────►│ Server │◄─────────│  Bob   │     │ Alice  │◄────────────►│  Bob   │
└────────┘      └────────┘          └────────┘     └────────┘              └────────┘
                                                   
All data flows through server!       Data flows DIRECTLY between peers!
Server = bottleneck + cost          Server only helps establish connection
```

The key insight: **video/audio/data can flow directly between users**, reducing server load and latency!

## 🔄 How WebRTC Works

WebRTC connection setup involves three main components:

### 1. Signaling Server
Helps peers find each other and exchange connection information.

### 2. STUN Server
**Session Traversal Utilities for NAT** - Helps discover your public IP address.

### 3. TURN Server  
**Traversal Using Relays around NAT** - Fallback relay when direct connection fails.

In [1]:
# Visualize WebRTC connection process

print("🔄 WebRTC Connection Process")
print("="*60)
print("""
Step 1: SIGNALING (via your server)
──────────────────────────────────────────────────────────────
┌───────┐                ┌───────────────┐                ┌───────┐
│ Alice │◄──WebSocket──►│   Signaling   │◄──WebSocket───►│  Bob  │
└───────┘                │    Server     │                └───────┘
                         └───────────────┘
                         
1. Alice creates "offer" (SDP - Session Description Protocol)
2. Alice sends offer to Bob via signaling server
3. Bob creates "answer" (SDP)
4. Bob sends answer to Alice via signaling server


Step 2: ICE CANDIDATE EXCHANGE
──────────────────────────────────────────────────────────────
┌───────┐        ┌──────────┐         
│ Alice │───────►│   STUN   │  "What's my public IP?"
└───────┘        │  Server  │  "You're at 203.0.113.1:5000"
                 └──────────┘
                 
Both peers discover their network addresses (ICE candidates)
and exchange them via signaling server.


Step 3: CONNECTION (direct or via TURN)
──────────────────────────────────────────────────────────────
Option A: Direct Connection (best case)
┌───────┐◄──────────────────────────────────────────►┌───────┐
│ Alice │          Direct P2P Connection            │  Bob  │
└───────┘                                            └───────┘

Option B: Via TURN Relay (when firewalls block direct)
┌───────┐◄──────►┌──────────┐◄──────►┌───────┐
│ Alice │        │   TURN   │        │  Bob  │
└───────┘        │  Server  │        └───────┘
                 └──────────┘
""")

🔄 WebRTC Connection Process

Step 1: SIGNALING (via your server)
──────────────────────────────────────────────────────────────
┌───────┐                ┌───────────────┐                ┌───────┐
│ Alice │◄──WebSocket──►│   Signaling   │◄──WebSocket───►│  Bob  │
└───────┘                │    Server     │                └───────┘
                         └───────────────┘

1. Alice creates "offer" (SDP - Session Description Protocol)
2. Alice sends offer to Bob via signaling server
3. Bob creates "answer" (SDP)
4. Bob sends answer to Alice via signaling server


Step 2: ICE CANDIDATE EXCHANGE
──────────────────────────────────────────────────────────────
┌───────┐        ┌──────────┐         
│ Alice │───────►│   STUN   │  "What's my public IP?"
└───────┘        │  Server  │  "You're at 203.0.113.1:5000"
                 └──────────┘

Both peers discover their network addresses (ICE candidates)
and exchange them via signaling server.


Step 3: CONNECTION (direct or via TURN)
───────────

## 🧱 Key Concepts

### SDP (Session Description Protocol)

SDP describes the media capabilities of each peer:
- What codecs are supported
- Media types (audio, video, data)
- Encryption keys

In [2]:
# Example SDP structure (simplified)

example_sdp = """
v=0
o=- 123456789 2 IN IP4 127.0.0.1
s=-
t=0 0
m=audio 49170 RTP/AVP 0
a=rtpmap:0 PCMU/8000
m=video 51372 RTP/AVP 99
a=rtpmap:99 H264/90000
"""

print("📄 Example SDP (Session Description Protocol)")
print("="*50)
print(example_sdp)
print("\n💡 SDP tells the other peer:")
print("   - I can do audio on port 49170")
print("   - I can do video on port 51372")
print("   - I support H264 codec for video")

📄 Example SDP (Session Description Protocol)

v=0
o=- 123456789 2 IN IP4 127.0.0.1
s=-
t=0 0
m=audio 49170 RTP/AVP 0
a=rtpmap:0 PCMU/8000
m=video 51372 RTP/AVP 99
a=rtpmap:99 H264/90000


💡 SDP tells the other peer:
   - I can do audio on port 49170
   - I can do video on port 51372
   - I support H264 codec for video


### ICE (Interactive Connectivity Establishment)

ICE finds the best way to connect peers. It tries multiple "candidates":

1. **Host candidates** - Direct local network (same WiFi)
2. **Server reflexive** - Via STUN (your public IP)
3. **Relay candidates** - Via TURN (last resort)

In [3]:
# ICE candidate types

print("🧊 ICE Candidate Types")
print("="*50)
print("""
1. HOST CANDIDATE (Best - Direct)
   ───────────────────────────────
   Your local IP address
   Works when: Same network (e.g., same office)
   Latency: Lowest possible!
   
   Example: 192.168.1.100:5000

2. SERVER REFLEXIVE (Good - Via STUN)
   ───────────────────────────────────
   Your public IP from STUN server
   Works when: No restrictive firewall
   Latency: Low (direct after discovery)
   
   Example: 203.0.113.1:40000

3. RELAY CANDIDATE (Fallback - Via TURN)
   ─────────────────────────────────────
   TURN server's address
   Works when: Direct blocked by firewall
   Latency: Higher (all traffic through TURN)
   Cost: TURN bandwidth costs money!
   
   Example: turn.example.com:3478
""")

🧊 ICE Candidate Types

1. HOST CANDIDATE (Best - Direct)
   ───────────────────────────────
   Your local IP address
   Works when: Same network (e.g., same office)
   Latency: Lowest possible!

   Example: 192.168.1.100:5000

2. SERVER REFLEXIVE (Good - Via STUN)
   ───────────────────────────────────
   Your public IP from STUN server
   Works when: No restrictive firewall
   Latency: Low (direct after discovery)

   Example: 203.0.113.1:40000

3. RELAY CANDIDATE (Fallback - Via TURN)
   ─────────────────────────────────────
   TURN server's address
   Works when: Direct blocked by firewall
   Latency: Higher (all traffic through TURN)
   Cost: TURN bandwidth costs money!

   Example: turn.example.com:3478



## 🌐 NAT Traversal: The Challenge

Most devices are behind NAT (Network Address Translation), which blocks incoming connections:

In [4]:
# NAT problem visualization

print("🔒 The NAT Problem")
print("="*60)
print("""
Alice's Home Network:                Bob's Home Network:
                                     
┌─────────────────────┐             ┌─────────────────────┐
│  Private: 192.168.1.5│             │  Private: 192.168.1.10│
│                     │             │                     │
│  ┌───────┐          │             │          ┌───────┐  │
│  │ Alice │          │             │          │  Bob  │  │
│  └───────┘          │             │          └───────┘  │
│        │            │             │            │        │
│        ▼            │             │            ▼        │
│  ┌──────────┐       │             │       ┌──────────┐  │
│  │  Router  │       │             │       │  Router  │  │
│  │   NAT    │       │             │       │   NAT    │  │
│  └────┬─────┘       │             │       └────┬─────┘  │
└───────┼─────────────┘             └────────────┼────────┘
        │                                        │
        ▼                                        ▼
   Public: 203.0.113.1                   Public: 198.51.100.5
        │                                        │
        └───────────── INTERNET ─────────────────┘

PROBLEM: Alice can't directly connect to Bob's private IP!
         Bob's router blocks incoming connections.
         
SOLUTION: STUN/TURN help "punch holes" through NAT!
""")

🔒 The NAT Problem

Alice's Home Network:                Bob's Home Network:

┌─────────────────────┐             ┌─────────────────────┐
│  Private: 192.168.1.5│             │  Private: 192.168.1.10│
│                     │             │                     │
│  ┌───────┐          │             │          ┌───────┐  │
│  │ Alice │          │             │          │  Bob  │  │
│  └───────┘          │             │          └───────┘  │
│        │            │             │            │        │
│        ▼            │             │            ▼        │
│  ┌──────────┐       │             │       ┌──────────┐  │
│  │  Router  │       │             │       │  Router  │  │
│  │   NAT    │       │             │       │   NAT    │  │
│  └────┬─────┘       │             │       └────┬─────┘  │
└───────┼─────────────┘             └────────────┼────────┘
        │                                        │
        ▼                                        ▼
   Public: 203.0.113.1                

## 🎥 WebRTC Use Cases

In [5]:
# WebRTC use cases

print("🎯 WebRTC Use Cases")
print("="*60)
print("""
✅ PERFECT FOR:
───────────────
🎥 Video Conferencing
   - Zoom, Google Meet, Microsoft Teams
   - Low latency is critical
   - Server bandwidth would be expensive!

📞 Voice Calls
   - Discord, Slack calls
   - Real-time audio needs low latency

🖥️ Screen Sharing
   - Collaborative debugging
   - Remote presentations

🎮 P2P Gaming
   - Real-time multiplayer
   - Reduce server costs

📝 Collaborative Editing (Advanced)
   - Google Docs uses WebRTC for presence
   - Direct sync between users (with CRDTs)


⚠️ PROBABLY OVERKILL FOR:
──────────────────────────
💬 Simple chat (WebSocket is simpler)
📊 Dashboards (SSE works fine)
🔔 Notifications (don't need P2P)
""")

🎯 WebRTC Use Cases

✅ PERFECT FOR:
───────────────
🎥 Video Conferencing
   - Zoom, Google Meet, Microsoft Teams
   - Low latency is critical
   - Server bandwidth would be expensive!

📞 Voice Calls
   - Discord, Slack calls
   - Real-time audio needs low latency

🖥️ Screen Sharing
   - Collaborative debugging
   - Remote presentations

🎮 P2P Gaming
   - Real-time multiplayer
   - Reduce server costs

📝 Collaborative Editing (Advanced)
   - Google Docs uses WebRTC for presence
   - Direct sync between users (with CRDTs)


⚠️ PROBABLY OVERKILL FOR:
──────────────────────────
💬 Simple chat (WebSocket is simpler)
📊 Dashboards (SSE works fine)
🔔 Notifications (don't need P2P)



## 📊 WebRTC vs WebSocket

In [6]:
# Comparison

print("📊 WebRTC vs WebSocket")
print("="*60)
print("""
                        WebSocket           WebRTC
────────────────────────────────────────────────────────────
Connection              Client ↔ Server     Client ↔ Client
Setup Complexity        Low                 High
Infrastructure          Simple server       Signaling + STUN/TURN
Best for               Chat, updates        Video, audio, P2P data
Latency                 Low                 Lowest (direct)
Server Load            All traffic          Minimal (signaling only)
Binary Support          Yes                 Yes
Browser Support         Excellent           Good

────────────────────────────────────────────────────────────

When to use WebSocket:
• You need a central server (chat room, game server)
• Simpler to implement
• All messages need to be recorded/processed

When to use WebRTC:
• Video/audio calls
• Reduce server bandwidth costs
• Need lowest possible latency
• Privacy (data doesn't go through server)
""")

📊 WebRTC vs WebSocket

                        WebSocket           WebRTC
────────────────────────────────────────────────────────────
Connection              Client ↔ Server     Client ↔ Client
Setup Complexity        Low                 High
Infrastructure          Simple server       Signaling + STUN/TURN
Best for               Chat, updates        Video, audio, P2P data
Latency                 Low                 Lowest (direct)
Server Load            All traffic          Minimal (signaling only)
Binary Support          Yes                 Yes
Browser Support         Excellent           Good

────────────────────────────────────────────────────────────

When to use WebSocket:
• You need a central server (chat room, game server)
• Simpler to implement
• All messages need to be recorded/processed

When to use WebRTC:
• Video/audio calls
• Reduce server bandwidth costs
• Need lowest possible latency
• Privacy (data doesn't go through server)



## 🏗️ WebRTC Architecture

A typical WebRTC system still needs some server infrastructure:

In [7]:
# WebRTC architecture

print("🏗️ WebRTC Architecture")
print("="*60)
print("""
                     ┌────────────────────────────────────┐
                     │         YOUR INFRASTRUCTURE        │
                     │                                    │
                     │  ┌───────────────────────────────┐ │
                     │  │      Signaling Server         │ │
                     │  │   (WebSocket or HTTP)         │ │
                     │  └───────────────────────────────┘ │
                     │               │                    │
                     │     Coordinates peer discovery     │
                     │     Exchanges SDP and ICE          │
                     └───────────────┬────────────────────┘
                                     │
           ┌─────────────────────────┼─────────────────────────┐
           │                         │                         │
           ▼                         │                         ▼
    ┌────────────┐                   │                  ┌────────────┐
    │   STUN     │                   │                  │   TURN     │
    │  Server    │                   │                  │  Server    │
    │ (free!)    │                   │                  │ (costly!)  │
    └────────────┘                   │                  └────────────┘
           │                         │                         │
           │      NAT Discovery      │      Relay Fallback     │
           │                         │                         │
           │                         │                         │
           │         ┌───────────────┴───────────────┐         │
           │         │                               │         │
           ▼         ▼                               ▼         ▼
         ┌─────────────────┐                   ┌─────────────────┐
         │     Alice       │◄═══════════════►│      Bob         │
         │   (Browser)     │  Direct P2P      │   (Browser)      │
         │                 │  or via TURN     │                  │
         └─────────────────┘                  └─────────────────┘

Cost Consideration:
──────────────────
• STUN: Almost free (just discovery)
• TURN: Expensive! All traffic flows through it
• ~10-20% of connections need TURN
""")

🏗️ WebRTC Architecture

                     ┌────────────────────────────────────┐
                     │         YOUR INFRASTRUCTURE        │
                     │                                    │
                     │  ┌───────────────────────────────┐ │
                     │  │      Signaling Server         │ │
                     │  │   (WebSocket or HTTP)         │ │
                     │  └───────────────────────────────┘ │
                     │               │                    │
                     │     Coordinates peer discovery     │
                     │     Exchanges SDP and ICE          │
                     └───────────────┬────────────────────┘
                                     │
           ┌─────────────────────────┼─────────────────────────┐
           │                         │                         │
           ▼                         │                         ▼
    ┌────────────┐                   │                  ┌────────────┐
    │   STU

## 🎯 System Design Interview Tips

In [8]:
# Interview tips

print("🎯 WebRTC in System Design Interviews")
print("="*60)
print("""
WHEN TO MENTION WebRTC:
────────────────────────
✅ "Design a video conferencing system like Zoom"
   → WebRTC is THE answer!
   
✅ "Design a screen sharing tool"
   → WebRTC for the media stream
   
✅ "Design Google Docs" (advanced)
   → Can mention WebRTC for presence/cursors


KEY POINTS TO MENTION:
───────────────────────
1. "WebRTC enables peer-to-peer communication"
2. "We need a signaling server for coordination"
3. "STUN helps with NAT traversal, TURN is the fallback"
4. "~10-20% of connections need TURN relay"
5. "This reduces server bandwidth costs significantly"


COMMON FOLLOW-UPS:
──────────────────
Q: "What if P2P fails?"
A: "TURN server acts as a relay - traffic goes through it"

Q: "How do peers find each other?"
A: "Signaling server (usually WebSocket) exchanges SDP"

Q: "What about group calls?"
A: "Options: Mesh (all peers connected) or SFU (Selective
    Forwarding Unit - server receives and distributes)"
""")

🎯 WebRTC in System Design Interviews

WHEN TO MENTION WebRTC:
────────────────────────
✅ "Design a video conferencing system like Zoom"
   → WebRTC is THE answer!

✅ "Design a screen sharing tool"
   → WebRTC for the media stream

✅ "Design Google Docs" (advanced)
   → Can mention WebRTC for presence/cursors


KEY POINTS TO MENTION:
───────────────────────
1. "WebRTC enables peer-to-peer communication"
2. "We need a signaling server for coordination"
3. "STUN helps with NAT traversal, TURN is the fallback"
4. "~10-20% of connections need TURN relay"
5. "This reduces server bandwidth costs significantly"


COMMON FOLLOW-UPS:
──────────────────
Q: "What if P2P fails?"
A: "TURN server acts as a relay - traffic goes through it"

Q: "How do peers find each other?"
A: "Signaling server (usually WebSocket) exchanges SDP"

Q: "What about group calls?"
A: "Options: Mesh (all peers connected) or SFU (Selective
    Forwarding Unit - server receives and distributes)"



## 👥 Group Calls: Mesh vs SFU

In [9]:
# Group call architectures

print("👥 Group Call Architectures")
print("="*60)
print("""
MESH (Pure P2P):
────────────────
Every peer connects to every other peer.

     ┌───────┐
     │ Alice │
     └───┬───┘
        ╱ ╲
       ╱   ╲
      ╱     ╲
 ┌───────┐ ┌───────┐
 │  Bob  │─│Charlie│
 └───────┘ └───────┘

Pros: No server bandwidth, low latency
Cons: Doesn't scale! N users = N*(N-1)/2 connections
Best for: 2-4 participants


SFU (Selective Forwarding Unit):
─────────────────────────────────
Server receives streams and forwards to others.

     ┌───────┐
     │ Alice │
     └───┬───┘
         │
         ▼
    ┌─────────┐
    │   SFU   │
    │ Server  │
    └────┬────┘
        ╱ ╲
       ╱   ╲
 ┌───────┐ ┌───────┐
 │  Bob  │ │Charlie│
 └───────┘ └───────┘

Pros: Scales to many participants, can add features
Cons: Server bandwidth cost, single point of failure
Best for: 5+ participants, enterprise features
""")

👥 Group Call Architectures

MESH (Pure P2P):
────────────────
Every peer connects to every other peer.

     ┌───────┐
     │ Alice │
     └───┬───┘
        ╱ ╲
       ╱   ╲
      ╱     ╲
 ┌───────┐ ┌───────┐
 │  Bob  │─│Charlie│
 └───────┘ └───────┘

Pros: No server bandwidth, low latency
Cons: Doesn't scale! N users = N*(N-1)/2 connections
Best for: 2-4 participants


SFU (Selective Forwarding Unit):
─────────────────────────────────
Server receives streams and forwards to others.

     ┌───────┐
     │ Alice │
     └───┬───┘
         │
         ▼
    ┌─────────┐
    │   SFU   │
    │ Server  │
    └────┬────┘
        ╱ ╲
       ╱   ╲
 ┌───────┐ ┌───────┐
 │  Bob  │ │Charlie│
 └───────┘ └───────┘

Pros: Scales to many participants, can add features
Cons: Server bandwidth cost, single point of failure
Best for: 5+ participants, enterprise features



## 🧪 Quick Quiz

1. **What is the role of a signaling server in WebRTC?**

2. **Why do we need TURN servers?**

3. **For a 1:1 video call app, would you use Mesh or SFU?**

In [10]:
# Quiz answers

print("📝 Quiz Answers")
print("="*50)
print("")
print("1. SIGNALING SERVER helps peers FIND each other!")
print("   It exchanges SDP (session info) and ICE candidates")
print("   (network addresses). It doesn't carry media traffic.")
print("")
print("2. TURN is needed when DIRECT connection fails!")
print("   Some firewalls block P2P. TURN acts as a relay")
print("   so traffic can still flow (through the server).")
print("")
print("3. MESH (direct P2P)!")
print("   For 1:1 calls, mesh is perfect - just one connection.")
print("   No need for SFU complexity with only 2 participants.")

📝 Quiz Answers

1. SIGNALING SERVER helps peers FIND each other!
   It exchanges SDP (session info) and ICE candidates
   (network addresses). It doesn't carry media traffic.

2. TURN is needed when DIRECT connection fails!
   Some firewalls block P2P. TURN acts as a relay
   so traffic can still flow (through the server).

3. MESH (direct P2P)!
   For 1:1 calls, mesh is perfect - just one connection.
   No need for SFU complexity with only 2 participants.


## ✅ Advantages of WebRTC

1. **Low latency** - Direct peer connection
2. **Reduced server costs** - No media relay
3. **Privacy** - Data doesn't go through servers
4. **Built into browsers** - No plugins needed
5. **Supports audio/video/data** - All-in-one

## ❌ Disadvantages

1. **Complex setup** - Signaling, STUN, TURN
2. **TURN costs** - 10-20% of connections need relay
3. **NAT issues** - Not always possible to connect
4. **Debugging hard** - Many moving parts
5. **Doesn't scale for groups** - Need SFU

## 📚 Summary

### What We Learned:

1. **WebRTC** = Peer-to-peer browser communication
2. **Signaling server** exchanges connection info (SDP)
3. **STUN** discovers public IP (NAT traversal)
4. **TURN** is a fallback relay when P2P fails
5. **Best for**: Video calls, screen sharing, P2P apps
6. **Group calls**: Mesh (small) or SFU (large)

### Interview Key Points:

> "For a video conferencing system, I'd use WebRTC for the media layer. It enables peer-to-peer connections which reduces server bandwidth costs. We'd need a signaling server (could use WebSocket) to exchange connection info, and STUN/TURN servers for NAT traversal. For group calls over 4-5 people, I'd recommend an SFU architecture."

### Next Up: Server-Side Push/Pull

Now that we've covered client-server protocols, let's explore how updates propagate from the **source to the server** - the second "hop" in our real-time system!